# Module 3 — Experimental Data Analysis with Pandas, siibra, and Nilearn

This notebook-style script combines tabular experimental data with atlas
information and ends by visualizing region-wise values on a brain template.

Learning goals:
- load and inspect tabular data with Pandas
- summarize values across experimental conditions
- work with region-level data
- fetch atlas information with siibra
- project region-wise values onto an atlas map
- visualize the result with Nilearn

Note:
This is a draft teaching notebook. Some siibra details may need adjustment
depending on the installed version and the chosen parcellation.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting
import siibra

# %matplotlib inline

## 1. Retrieve the labelled map for Julich Brain Atlas on MNI 152 reference space

In [ ]:
labelled_map = siibra.get_map(parcellation="julich 3.1", space="mni152", maptype="labelled")
labelled_map

In [ ]:
# download the map as a nifti
map_nii = labelled_map.fetch()
map_nii

## 2. Plot the atlas using nilearn

In [ ]:
# get the colormap to display the map
colormap = labelled_map.get_colormap(fill_uncolored=True)

In [ ]:
# plot interactively with nilarn
plotting.view_img(
    map_nii,
    title="Region-wise experimental values on atlas",
    cmap=colormap,
    symmetric_cmap=False
)

## 3. Synthetic data to mimic experiment results

Normally, there would be data you can read from a csv, txt, tsv and so on where recordings.



In [ ]:
# Random selection of regions from Julich Brain Atlas 
regions = [labelled_map.regions[i] for i in np.random.randint(0, len(labelled_map.regions), 30)]
tasks = ["rest", "motor task 1", "motor task 2", "visual task 1", "visual task 2", "visual task 3", "auditory task"]
experimental_data = pd.DataFrame(
    [{"region": r, **{t: np.random.randint(0, 250) for t in tasks}} for r in regions]
).set_index('region')
experimental_data

## 4. Basic statistics using pandas



In [ ]:
experimental_data.describe()

## 5. Basic plotting from the dataframe



In [ ]:
experimental_data.plot(kind='bar', figsize=(15, 10), grid="y-axis")

In [ ]:
experimental_data['rest'].plot(kind='bar', grid="y-axis")

In [ ]:
experimental_data[
    [col for col in experimental_data.columns if "visual" in col]
].plot(kind='bar')

## 6. Match experimental values to atlas regions

We create a simple lookup table from region name to value.



In [ ]:
value_lookup = dict(zip(experimental_data.index, experimental_data["rest"]))
colored_img = labelled_map.colorize(value_lookup).fetch()
plotting.view_img(
    colored_img,
    title="Region-wise experimental values on atlas",
    cmap='jet',
    symmetric_cmap=False
)

In [ ]:
# Alternative view: glass brain
plotting.plot_glass_brain(
    colored_img,
    title="Glass brain view of region-wise values",
    display_mode="lyrz",
    cmap="jet",
    threshold=0,
)
plotting.show()

## 7. Get BOLD data for the regions in the DataFrame and explore the data 

In [ ]:
bold_feat = siibra.features.get(labelled_map.parcellation, "BOLD")[0]
bold_feat.name

In [ ]:
for element in bold_feat.elements:
    print(element.subject)

## 8. Plot the BOLD data for all regions

In [ ]:
bold_feat.plot()

## 9. Display BOLD recordings only for the regions of interest

In [ ]:
selected_bold = bold_feat.data.loc[experimental_data.index.to_list()]
selected_bold

In [ ]:
selected_bold_std = selected_bold.std()
selected_bold_std

In [ ]:
# Alternative view: glass brain
plotting.plot_glass_brain(
    labelled_map.colorize(selected_bold_std.to_dict()).fetch(),
    title="Glass brain view of region-wise values",
    display_mode="lyrz",
    cmap="jet",
    threshold=0,
)
plotting.show()

## 10. Compute correlations

In [ ]:
def correlate_series(x, y, method="spearman", log_y=False):
    """
    Correlate two pandas Series with shared index labels.

    method: "pearson", "spearman", or "kendall"
    log_y: useful if y is variance/SD-like and strongly skewed
    """
    df = pd.concat([x.rename("x"), y.rename("y")], axis=1).dropna()

    if log_y:
        df["y"] = np.log1p(df["y"])

    if len(df) < 3:
        raise ValueError("Need at least 3 paired observations after alignment/dropna.")

    return df["x"].corr(df["y"], method=method)

In [ ]:
r = correlate_series(selected_bold_std, experimental_data["motor task 1"], method="spearman")
print(r)

## 11. Plot correlations

In [ ]:
plotting.plot_glass_brain(
    labelled_map.colorize(r.to_dict()).fetch(),
    title="Glass brain view of region-wise values",
    display_mode="lyrz",
    cmap="jet",
    threshold=0,
)
plotting.show()